In [1]:
import numpy as np
from mlp_weights import W1, B1, W2, B2, N_INPUT, N_HIDDEN, N_OUTPUT

# =========================================================
# tanh approximation (same polynomial as HLS)
# =========================================================
def tanh_approx(x):
    x = np.clip(x, -1.0, 1.0)
    C1 = 0.99
    C3 = 0.33
    return C1 * x - C3 * (x ** 3)

# =========================================================
# Single MLP evaluation
# =========================================================
def mlp_single(state):
    # Hidden layer
    hidden = np.zeros(N_HIDDEN, dtype=np.float32)
    for h in range(N_HIDDEN):
        acc = B1[h]
        for i in range(N_INPUT):
            acc += W1[h, i] * state[i]
        hidden[h] = tanh_approx(acc)

    # Output layer
    outv = np.zeros(N_OUTPUT, dtype=np.float32)
    for o in range(N_OUTPUT):
        acc = B2[o]
        for h in range(N_HIDDEN):
            acc += W2[o, h] * hidden[h]
        outv[o] = acc

    return outv

# =========================================================
# pack32 (same logic as HLS)
# =========================================================
def pack32(outv):
    r = np.uint32(0)
    for i in range(N_OUTPUT):
        bits = np.uint32(np.frombuffer(outv[i].tobytes(), dtype=np.uint32)[0])
        r ^= (bits & 0x3FF) << (10 * i)
    return r

# =========================================================
# ARX mixer
# =========================================================
def arx_mix(x):
    rot = ((x << 7) | (x >> 25)) & 0xFFFFFFFF
    return np.uint32(rot ^ (x + 0x9E3779B9))

# =========================================================
# TOP-LEVEL RNG (equivalent to mlp_rng_top)
# =========================================================
class MLPRNG:
    def __init__(self):
        self.state = np.zeros((4, N_INPUT), dtype=np.float32)
        self.idx = 0
        self.reset()

    def reset(self):
        for e in range(4):
            for i in range(N_INPUT):
                self.state[e, i] = 0.01 * (e + 1) * (i + 1)
        self.idx = 0

    def step(self):
        outv = mlp_single(self.state[self.idx])

        # feedback update
        self.state[self.idx] = outv[:N_INPUT]

        # RNG output
        rand_word = arx_mix(pack32(outv))

        self.idx = (self.idx + 1) & 3
        return rand_word

# =========================================================
# Example usage
# =========================================================
if __name__ == "__main__":
    rng = MLPRNG()

    words = []
    for _ in range(10000):
        words.append(rng.step())

    # Save to binary file
    with open("mlp_rng_output.bin", "wb") as f:
        for w in words:
            f.write(w.tobytes())

    print("Generated", len(words), "32-bit random words")


Generated 10000 32-bit random words
